In [18]:
import pandas as pd
import numpy as np

customers = pd.read_csv("../data/raw/customers.csv")

customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   customer_id  150 non-null    int64
 1   name         150 non-null    str  
 2   gender       150 non-null    str  
 3   age          150 non-null    int64
 4   city         150 non-null    str  
 5   signup_date  150 non-null    str  
dtypes: int64(2), str(4)
memory usage: 7.2 KB


In [ ]:

order_items = pd.read_csv("../data/raw/order_items.csv")

order_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 764 entries, 0 to 763
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   order_item_id  764 non-null    int64
 1   order_id       764 non-null    int64
 2   product_id     764 non-null    int64
 3   quantity       764 non-null    int64
 4   unit_price     764 non-null    int64
dtypes: int64(5)
memory usage: 30.0 KB


In [20]:
orders = pd.read_csv("../data/raw/orders.csv")

orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   order_id        300 non-null    int64
 1   customer_id     300 non-null    int64
 2   order_date      300 non-null    str  
 3   payment_method  300 non-null    str  
 4   order_status    300 non-null    str  
dtypes: int64(2), str(3)
memory usage: 11.8 KB


In [21]:
products = pd.read_csv("../data/raw/products.csv")

products.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   product_id    100 non-null    int64
 1   product_name  100 non-null    str  
 2   category      100 non-null    str  
 3   price         100 non-null    int64
dtypes: int64(2), str(2)
memory usage: 3.3 KB


In [22]:
# 날짜 타입
orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")
orders["order_date"].info()

<class 'pandas.Series'>
RangeIndex: 300 entries, 0 to 299
Series name: order_date
Non-Null Count  Dtype         
--------------  -----         
300 non-null    datetime64[us]
dtypes: datetime64[us](1)
memory usage: 2.5 KB


In [23]:
orders.head()

,order_id,customer_id,order_date,payment_method,order_status
0,1,123,2026-07-02,card,completed
1,2,77,2025-09-17,naver_pay,cancelled
2,3,138,2026-01-14,bank_transfer,cancelled
3,4,57,2026-03-27,kakao_pay,cancelled
4,5,125,2026-02-15,card,cancelled


In [26]:
# 키 결측 확인(부모 테이블 중 결측 확인)
print(customers["customer_id"].isna().sum())
print(order_items["order_item_id"].isna().sum())
print(orders["order_id"].isna().sum())
print(products["product_id"].isna().sum())

0
0
0
0


In [30]:
for frame, key in [
    (customers, "customer_id"),
    (order_items, "order_item_id"),
    (orders, "order_id"),
    (products, "product_id")
]:
    print(key, "결측 개수:", frame[key].isna().sum())

customer_id 결측 개수: 0
order_item_id 결측 개수: 0
order_id 결측 개수: 0
product_id 결측 개수: 0


단변량 EDA

In [32]:
# 도시(city), 성별(gender), 나이(age)로 등록 고객의 분포(value_counts, describe)를 확인해 보세요.
print(customers["city"].value_counts())
print(customers["gender"].value_counts())

print(customers["age"].describe())

city
성남    21
광주    17
부산    16
대구    15
서울    15
울산    14
인천    14
대전    14
수원    13
고양    11
Name: count, dtype: int64
gender
F    84
M    66
Name: count, dtype: int64
count    150.000000
mean      42.086667
std       15.613166
min       19.000000
25%       29.000000
50%       40.000000
75%       57.000000
max       69.000000
Name: age, dtype: float64


In [43]:
# 상품 카테고리의 개수와 가격 통계
product_count = products["category"].value_counts(dropna=False).to_frame()
type(product_count)

pandas.DataFrame

In [34]:
products["price"].describe()

count       100.000000
mean     110040.000000
std       56433.910574
min        5000.000000
25%       65750.000000
50%      112000.000000
75%      161000.000000
max      200000.000000
Name: price, dtype: float64

In [36]:
# 상품 카테고리별 분포
catetory_price = products.groupby("category", dropna=False).agg(
    product_count = ("product_id", "size"),
    mean_price = ("price", "mean"),
    median_price = ("price", "median")
)

print(catetory_price)

          product_count     mean_price  median_price
category                                            
도서                   14  106857.142857      118500.0
뷰티                   16  117687.500000      134000.0
생활용품                 16   96437.500000       89000.0
스포츠                  19  111578.947368      103000.0
식품                    7  137142.857143      147000.0
전자기기                 17  101588.235294      111000.0
패션                   11  115909.090909      115000.0


완료 주문 병합 후 검증

In [37]:
# 완료된 주문건 조회
# 불리언 마스크
# orders["order_status"] == "completed"
completed_orders = orders[orders["order_status"] == "completed"]
print(completed_orders["order_status"].value_counts())

order_status
completed    184
Name: count, dtype: int64


In [ ]:
# order_items 에서 completed 인 항목

items_with_orders = order_items.merge(
    orders[["order_id", "customer_id", "order_date", "order_status"]],
    on="order_id", how="left", validate="many_to_one", indicator=True,
)

items_with_orders.head()

,order_item_id,order_id,product_id,quantity,unit_price,customer_id,order_date,order_status,_merge
0,1,1,100,3,102000,123,2026-07-02,completed,both
1,2,1,87,5,25000,123,2026-07-02,completed,both
2,3,1,7,3,142000,123,2026-07-02,completed,both
3,4,1,9,3,193000,123,2026-07-02,completed,both
4,5,2,72,4,189000,77,2025-09-17,cancelled,both


In [47]:
# 고객별 평균 주문 건수(완료건 기준)
completed_orders = orders[orders["order_status"] == "completed"]
completed_orders.head()

,order_id,customer_id,order_date,payment_method,order_status
0,1,123,2026-07-02,card,completed
5,6,87,2026-05-16,naver_pay,completed
8,9,145,2026-03-17,card,completed
10,11,47,2026-01-12,card,completed
11,12,98,2025-09-06,kakao_pay,completed


In [46]:
# 카테고리 별 수량과 완료 주문 금액을 합산 해 보세요
# 카테고리(products), 주문상태 완료(orders), 금액(order_items)
# orders.head()
order_items_with_orders = order_items.merge(
    orders[["order_id","order_status"]],
    on="order_id",
    how="left",
    validate="many_to_one"
)